In [1]:
# ============================================================
# CELL 1: Setup + Install
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

!pip install gradio transformers peft pytesseract sentencepiece protobuf -q
!apt-get -qq install tesseract-ocr

import os, torch

class Config:
    PROJECT_ROOT = '/content/drive/MyDrive/FinDocVQA'
    MODELS_DIR   = os.path.join(PROJECT_ROOT, 'models')

cfg = Config()
print(f"✓ GPU: {torch.cuda.get_device_name(0)}")

Mounted at /content/drive
✓ GPU: Tesla T4


In [2]:
# ============================================================
# CELL 2: Load Models
# ============================================================

from transformers import (
    Pix2StructProcessor, Pix2StructForConditionalGeneration,
    LayoutLMv3Processor, LayoutLMv3ForQuestionAnswering,
    pipeline
)
import pytesseract
from PIL import Image
import re

# --- Pix2Struct ---
print("Loading Pix2Struct...")
p2s_processor = Pix2StructProcessor.from_pretrained('google/pix2struct-docvqa-base')
p2s_model = Pix2StructForConditionalGeneration.from_pretrained('google/pix2struct-docvqa-base')
p2s_model = p2s_model.to('cuda').eval()
print("✓ Pix2Struct loaded")

# --- LayoutLMv3 ---
print("Loading LayoutLMv3...")
lmv3_processor = LayoutLMv3Processor.from_pretrained('rubentito/layoutlmv3-base-mpdocvqa', apply_ocr=True)
lmv3_model = LayoutLMv3ForQuestionAnswering.from_pretrained('rubentito/layoutlmv3-base-mpdocvqa')
lmv3_model = lmv3_model.to('cuda').eval()
print("✓ LayoutLMv3 loaded")

# --- OCR + RoBERTa ---
print("Loading RoBERTa baseline...")
qa_pipeline = pipeline("question-answering", model="deepset/roberta-base-squad2", device=0)
print("✓ RoBERTa loaded")

print("\n✓ All models ready")

Loading Pix2Struct...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/249 [00:00<?, ?B/s]

The image processor of type `Pix2StructImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/79.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

✓ RoBERTa loaded

✓ All models ready


In [3]:
# ============================================================
# CELL 3: Define prediction functions
# ============================================================

def predict_pix2struct(image, question):
    image = image.convert('RGB')
    inputs = p2s_processor(images=image, text=question, return_tensors="pt", max_patches=2048).to('cuda')
    with torch.no_grad():
        generated = p2s_model.generate(**inputs, max_new_tokens=256)
    return p2s_processor.decode(generated[0], skip_special_tokens=True)

def predict_layoutlmv3(image, question):
    image = image.convert('RGB')
    encoding = lmv3_processor(image, question, return_tensors="pt",
                               max_length=512, truncation=True, padding="max_length")
    encoding = {k: v.to('cuda') for k, v in encoding.items()}
    with torch.no_grad():
        outputs = lmv3_model(**encoding)
    start_idx = torch.argmax(outputs.start_logits, dim=1).item()
    end_idx = torch.argmax(outputs.end_logits, dim=1).item()
    if end_idx < start_idx:
        end_idx = start_idx
    input_ids = encoding['input_ids'][0]
    answer_tokens = input_ids[start_idx : end_idx + 1]
    return lmv3_processor.tokenizer.decode(answer_tokens, skip_special_tokens=True).strip()

def predict_ocr_roberta(image, question):
    image = image.convert('RGB')
    ocr_text = pytesseract.image_to_string(image)
    if not ocr_text.strip():
        return "(no text detected)"
    try:
        result = qa_pipeline(question=question, context=ocr_text[:4000])
        return result['answer']
    except:
        return "(error)"

In [4]:
# ============================================================
# CELL 4: Build and Launch Gradio App
# ============================================================

import gradio as gr

def financial_vqa(image, question):
    """Run all three models and return results."""
    if image is None or not question.strip():
        return "No image", "No image", "No image"

    image = Image.fromarray(image) if not isinstance(image, Image.Image) else image

    try:
        p2s_answer = predict_pix2struct(image, question)
    except Exception as e:
        p2s_answer = f"Error: {str(e)[:50]}"

    try:
        lmv3_answer = predict_layoutlmv3(image, question)
    except Exception as e:
        lmv3_answer = f"Error: {str(e)[:50]}"

    try:
        roberta_answer = predict_ocr_roberta(image, question)
    except Exception as e:
        roberta_answer = f"Error: {str(e)[:50]}"

    return p2s_answer, lmv3_answer, roberta_answer

# Sample questions for quick testing
examples = [
    ["What was the total net sales?"],
    ["What was the net income in 2025?"],
    ["Which line item appears below Cost of sales?"],
    ["By what percentage did revenue change?"],
    ["What type of chart is shown?"],
]

demo = gr.Interface(
    fn=financial_vqa,
    inputs=[
        gr.Image(type="pil", label="Upload Financial Document"),
        gr.Textbox(label="Question", placeholder="Ask a question about the document..."),
    ],
    outputs=[
        gr.Textbox(label="Pix2Struct Answer"),
        gr.Textbox(label="LayoutLMv3 Answer"),
        gr.Textbox(label="OCR + RoBERTa Answer"),
    ],
    title="Financial Document VQA",
    description=(
        "Upload a financial document image (SEC filing, earnings report, balance sheet) "
        "and ask a question. Three models will attempt to answer:\n\n"
        "• **Pix2Struct** — OCR-free vision-language model\n"
        "• **LayoutLMv3** — OCR + spatial layout model\n"
        "• **OCR + RoBERTa** — traditional OCR pipeline baseline\n\n"
        "Note: These models were trained on generic documents and struggle with "
        "financial documents — this demo illustrates the domain gap."
    ),
    examples=None,
    allow_flagging="never",
)

demo.launch(share=True, debug=True)

/usr/local/lib/python3.12/dist-packages/gradio/interface.py:415: UserWarning: The `allow_flagging` parameter in `Interface` is deprecated. Use `flagging_mode` instead.
  warnings.warn(


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://e9fe3cb8a87489ebd0.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Arial.TTF: 0.00B [00:00, ?B/s]

Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://e9fe3cb8a87489ebd0.gradio.live
